Damped Newton method for chi=30 truncating to 14,14

Start Newton method after 4 RG steps.

Damping with newton_step=0.5 is activated a couple of times

Here I start with gilt_eps = 2e-5, run until I reach error 1e-5, switch to gilt_eps = 6e-10


In [23]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [17]:
gilt_eps = 2e-5 #6e-6
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.03361407516010624 and became 0.0. Index CartesianIndex(1, 16, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.012345916839578114 and became -1.8524388175538877e-9. Index CartesianIndex(15, 2, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.004606248308129038 and became 0.0. Index CartesianIndex(17, 3, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0022689612252432567 and became 0.0. Index CartesianIndex(15, 2, 19, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It wa

In [18]:
for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

1 [1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
7 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
10 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
11 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]


In [19]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

In [22]:
A[1] = truncate_blocks(traj[4], trunc_shape)
Afin = A[1]
deltaAfin = A[1]
i=0
for i in 1:30
    println("i=",i)
    println("gilt_eps = ", gilt_pars["gilt_eps"])
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    Anew = A[i]
    accepted_elements_new = accepted_elements[i]
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
    #let's try to increase gilt_eps
    gilt_eps_try = max(gilt_pars["gilt_eps"] * 0.94, 6e-6)
    gilt_pars_try = Dict(
        	"gilt_eps" => gilt_eps_try,
        	"cg_chis" => collect(1:chi),
        	"cg_eps" => cg_eps,
        	"verbosity" => 1,
        	"rotate" => true
            )
    etry, _ = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars_try; trunc_shape = trunc_shape)
    println("tried gilt_eps =", gilt_eps_try)
    println("new error =",etry) 
    if etry < 1.5*enew
        gilt_pars = gilt_pars_try
        println("accepting new gilt_eps")
    end
end

i=1
gilt_eps = 2.0e-5
||R(A[i])-A[i]||= 0.0578969047685301
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  14 eigenvalues converged
│ *  norm of residuals = (1.3583986242457702e-54, 4.831605602993206e-40, 1.0019251957975019e-39, 1.4928941611843735e-28, 1.3593990829850754e-27, 1.6084510784456484e-23, 1.6084510784456484e-23, 4.612099180393743e-17, 6.944801565260803e-16, 6.944801565260803e-16, 4.614894945840612e-15, 4.614894945840612e-15, 6.936518121219281e-15, 6.936518121219281e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.985040503109203 + 0.0im
-0.935033225880193 + 0.0im
-0.925018244490617 + 0.0im
0.5571725104232076 + 0.0im
0.5510069815140766 + 0.0im
0.000705315559213195 + 0.41914598564475036im
0.000705315559213195 - 0.41914598564475036im
-0.31049865202182714 + 0.0im
6.853188824139413e-5 + 0.30966015717141404im
6.853188824139413e-5 - 0.30966015717141404im
0.07550650733644826 + 0.2741295040237229im
0.07550650733644826 - 0.2741295040237229im
-0.07538410144938851 + 0.2738541008863207im
-0.07538410144938851 - 0.2738541008863207im
||deltaA[i]||= 0.13403766378436696
newton_step= 1.0
fp_error= 0.027517711938467204
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
tried gilt_eps =1.88e-5
new error =0.028788288480504878
accepting new gilt_eps
i=2
gilt_eps = 1.88e-5
||R(A[i])-A[i]||= 0.028788288480504878
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  12 eigenvalues converged
│ *  norm of residuals = (3.4999024774368126e-53, 1.7313064486644615e-38, 9.128388611646307e-39, 5.020308377815047e-31, 1.2054614838297046e-28, 1.3962241413392784e-27, 1.3962241413392784e-27, 1.746162392003562e-21, 2.0471500761988817e-15, 2.0471500761988817e-15, 5.4019132044973825e-15, 5.4019132044973825e-15)
└ *  number of operations = 60


EIGENVALUES (INITIAL):
1.9885981739388727 + 0.0im
-0.9707715306418134 + 0.0im
-0.9594390111303249 + 0.0im
0.6455202035454691 + 0.0im
0.5893619561257583 + 0.0im
-0.002868732598623156 + 0.5313682958834313im
-0.002868732598623156 - 0.5313682958834313im
-0.4229587827918313 + 0.0im
0.014410306438174439 + 0.31268077458981547im
0.014410306438174439 - 0.31268077458981547im
0.11796233179245975 + 0.26117845246458843im
0.11796233179245975 - 0.26117845246458843im
||deltaA[i]||= 0.07893685458367879
newton_step= 1.0
fp_error= 0.11107886976631688
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
newton_step= 0.5
fp_error= 0.012208394678541994
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
tried gilt_eps =1.7672e-5
new error =0.01274933462872751
accepting new gilt_eps
i=3
gilt_eps = 1.7672e-5
||R(A[i])-A[i]||= 0.01274933462872751
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


...

┌ Info: Arnoldi eigsolve finished after 5 iterations:
│ *  19 eigenvalues converged
│ *  norm of residuals = (1.8227148145745388e-61, 4.592226280975622e-46, 4.6133017719274354e-46, 2.2648311326115256e-41, 6.682701098823028e-35, 1.6724085728268344e-33, 1.6724085728268344e-33, 1.1455671942290035e-21, 2.858651126263013e-15, 3.1930267063728626e-18, 3.1930267063728626e-18, 5.315589554753925e-17, 5.315589554753925e-17, 2.8044096496863852e-14, 2.8044096496863852e-14, 9.315166789031215e-18, 9.315166789031215e-18, 1.3945531988716353e-14, 1.3945531988716353e-14)
└ *  number of operations = 68


EIGENVALUES (INITIAL):
1.997855466404059 + 0.0im
-0.9884500085773555 + 0.0im
-0.9841506280109755 + 0.0im
0.7432718285093711 + 0.0im
0.6001968889950614 + 0.0im
-0.0037837427998667476 + 0.5328087952902844im
-0.0037837427998667476 - 0.5328087952902844im
-0.36808871313084324 + 0.0im
0.29001897246036135 + 0.0im
-0.15381900749229854 + 0.24064728667761026im
-0.15381900749229854 - 0.24064728667761026im
-0.00927567901115901 + 0.2784079452837194im
-0.00927567901115901 - 0.2784079452837194im
0.2761232590333288 + 0.03059866346105547im
0.2761232590333288 - 0.03059866346105547im
0.13985617920519056 + 0.2393281321712881im
0.13985617920519056 - 0.2393281321712881im
0.014478704649809618 + 0.25990498071338286im
0.014478704649809618 - 0.25990498071338286im
||deltaA[i]||= 0.0047494009680951405
newton_step= 1.0
fp_error= 0.0003503702448147426
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
tried gilt_eps =1.6611679999999997e-5
new error =0.10536835620043287
i=10
gilt_eps = 1.7672e-5
||R(A[i

┌ Info: Arnoldi eigsolve finished after 5 iterations:
│ *  19 eigenvalues converged
│ *  norm of residuals = (7.021073701377749e-61, 1.526634791227313e-45, 2.423147735774316e-46, 1.82292532296149e-40, 2.476153292597989e-34, 5.097539965871376e-34, 5.097539965871376e-34, 2.4350517634606703e-26, 1.8809457223630143e-14, 1.2661555460050503e-18, 1.2661555460050503e-18, 8.21005154407975e-18, 8.21005154407975e-18, 1.2136833172966029e-13, 1.2136833172966029e-13, 8.688961874390602e-17, 8.688961874390602e-17, 1.8339539831695384e-15, 1.8339539831695384e-15)
└ *  number of operations = 68


EIGENVALUES (INITIAL):
1.9979413612475585 + 0.0im
-0.9916621229995435 + 0.0im
-0.9856430486789448 + 0.0im
0.7173207165882053 + 0.0im
0.5980090865142952 + 0.0im
0.0035461469070345976 + 0.5551872951654507im
0.0035461469070345976 - 0.5551872951654507im
-0.4147826590271731 + 0.0im
0.2897502001415668 + 0.0im
-0.14747647067746822 + 0.24387693495426985im
-0.14747647067746822 - 0.24387693495426985im
0.12954199927393242 + 0.2496180855809915im
0.12954199927393242 - 0.2496180855809915im
0.27675827028033106 + 0.031552306792778005im
0.27675827028033106 - 0.031552306792778005im
-0.017027962729550233 + 0.2767328261912416im
-0.017027962729550233 - 0.2767328261912416im
0.020801225499672144 + 0.2635723917599659im
0.020801225499672144 - 0.2635723917599659im
||deltaA[i]||= 0.0007158512521151496
newton_step= 1.0
fp_error= 0.00023005011607573906
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
tried gilt_eps =1.6611679999999997e-5
new error =0.10537627218667389
i=11
gilt_eps = 1.7672e-5
||R(A

LoadError: InterruptException: